# LSTM — Paper-to-Code Mock (Colab)

**Paper:** Long Short-Term Memory (Hochreiter & Schmidhuber, 1997), *Neural Computation* 9(8):1735–1780 — https://www.bioinf.jku.at/publications/older/2604.pdf  •  Explainer: https://colah.github.io/posts/2015-08-Understanding-LSTMs/

Medium–hard mock (~60 min). Fill in the `LSTMCell` stub, run the long-range memory demo (LSTM vs vanilla RNN), then the sanity checks. Reference solution in the last cell.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## 1. Implement the LSTM cell (YOUR TASK)

From `[h, x]`: input gate `i=σ(...)`, forget gate `f=σ(...)`, output gate `o=σ(...)`, candidate `g=tanh(...)`.
Additive cell update `c' = f⊙c + i⊙g`; hidden `h' = o⊙tanh(c')`. Init the forget-gate bias positive so the cell defaults to remembering.

`RNNCell` and `SeqModel` are given for the comparison.

In [ ]:
class LSTMCell(nn.Module):
    def __init__(self, in_dim, hid_dim, forget_bias=1.0):
        super().__init__()
        self.hid_dim = hid_dim
        self.W = nn.Linear(in_dim + hid_dim, 4 * hid_dim)
        # TODO: with torch.no_grad(): set self.W.bias[hid_dim:2*hid_dim] = forget_bias
        raise NotImplementedError("set the forget-gate bias")

    def forward(self, x, state):
        h, c = state
        # TODO: z = self.W(cat([h, x]))  ->  chunk into i, f, g, o
        # TODO: i,f,o = sigmoid;  g = tanh
        # TODO: c_new = f*c + i*g;  h_new = o*tanh(c_new)
        # TODO: return h_new, c_new
        raise NotImplementedError("fill me in")


class RNNCell(nn.Module):
    """Vanilla tanh RNN step (given): h' = tanh(W[h, x] + b)."""
    def __init__(self, in_dim, hid_dim):
        super().__init__()
        self.hid_dim = hid_dim
        self.W = nn.Linear(in_dim + hid_dim, hid_dim)

    def forward(self, x, state):
        h, _ = state
        h_new = torch.tanh(self.W(torch.cat([h, x], dim=-1)))
        return h_new, h_new


class SeqModel(nn.Module):
    """Loops a cell over a sequence, reads out from the final hidden state (given)."""
    def __init__(self, cell_cls, in_dim, hid_dim, out_dim):
        super().__init__()
        self.cell = cell_cls(in_dim, hid_dim)
        self.hid_dim = hid_dim
        self.readout = nn.Linear(hid_dim, out_dim)

    def forward(self, x):
        B, T, _ = x.shape
        h = x.new_zeros(B, self.hid_dim)
        c = x.new_zeros(B, self.hid_dim)
        for t in range(T):
            h, c = self.cell(x[:, t, :], (h, c))
        return self.readout(h)

## 2. Demonstrate the benefit (long-range memory: delayed XOR)
Two cue bits at t=0 and t=1, then T-2 steps of noise; target = XOR of the two bits. The net must hold both bits across the gap and combine them. Train an LSTM and a vanilla RNN on the same data.

In [ ]:
def make_xor_data(n, T, seed):
    g = torch.Generator().manual_seed(seed)
    x = torch.zeros(n, T, 2)
    a = (torch.rand(n, generator=g) < 0.5).float()
    b = (torch.rand(n, generator=g) < 0.5).float()
    x[:, 0, 0] = a * 2 - 1                                # first cue bit
    x[:, 1, 0] = b * 2 - 1                                # second cue bit
    x[:, 2:, 1] = torch.randn(n, T - 2, generator=g)      # distractor noise
    y = (a.long() ^ b.long())                            # XOR target
    return x, y

def train_model(cell_cls, T, hid=32, steps=600, lr=3e-3, seed=0):
    torch.manual_seed(seed)
    Xtr, ytr = make_xor_data(256, T, seed=1)
    Xte, yte = make_xor_data(512, T, seed=2)
    model = SeqModel(cell_cls, 2, hid, 2)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(steps):
        model.train()
        loss = F.cross_entropy(model(Xtr), ytr)
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        acc = (model(Xte).argmax(-1) == yte).float().mean().item()
    return model, acc

T = 60
rnn_model, rnn_acc = train_model(RNNCell, T, seed=0)
lstm_model, lstm_acc = train_model(LSTMCell, T, seed=0)
print(f"vanilla RNN : test acc {rnn_acc:.3f}")   # ~0.49 (chance)
print(f"LSTM        : test acc {lstm_acc:.3f}")   # ~1.00 (solved)

## 3. Sanity checks

In [ ]:
# 1) output shapes
cell = LSTMCell(2, 8)
h1, c1 = cell(torch.randn(4, 2), (torch.zeros(4, 8), torch.zeros(4, 8)))
assert h1.shape == (4, 8) and c1.shape == (4, 8)

# 2) gates in (0,1), candidate in (-1,1)
cell = LSTMCell(2, 16)
z = cell.W(torch.cat([torch.randn(100, 16), torch.randn(100, 2)], -1))
i, f, g, o = z.chunk(4, -1)
i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o); g = torch.tanh(g)
assert i.min() > 0 and i.max() < 1 and f.min() > 0 and f.max() < 1 and o.min() > 0 and o.max() < 1
assert g.min() > -1 and g.max() < 1

# 3) constant error carousel: forget=1, input=0 -> cell unchanged
cell = LSTMCell(2, 8); H = 8
with torch.no_grad():
    cell.W.weight.zero_(); cell.W.bias.zero_()
    cell.W.bias[0:H] = -50.0; cell.W.bias[H:2*H] = 50.0
c_prev = torch.randn(3, 8)
_, c_next = cell(torch.randn(3, 2), (torch.randn(3, 8), c_prev))
assert torch.allclose(c_next, c_prev, atol=1e-5)

# 4) LSTM solves long-range task; vanilla RNN fails
assert lstm_acc > 0.9 and rnn_acc < 0.75 and (lstm_acc - rnn_acc) > 0.2

# 5) gradient flows to the earliest input (non-vanishing vs RNN)
Tg = 60
def early_grad(cell_cls, seed=0):
    torch.manual_seed(seed)
    m = SeqModel(cell_cls, 2, 16, 2)
    x = torch.randn(8, Tg, 2, requires_grad=True)
    m(x).sum().backward()
    return x.grad[:, 0, :].abs().mean().item()
g_lstm, g_rnn = early_grad(LSTMCell), early_grad(RNNCell)
print(f"early-input grad  LSTM {g_lstm:.3e}   RNN {g_rnn:.3e}")
assert g_lstm > g_rnn * 10

# 6) full-sequence forward is finite and right-shaped
out = lstm_model(torch.randn(5, T, 2))
assert out.shape == (5, 2) and torch.isfinite(out).all()

print("All sanity checks passed.")

## 4. Reference solution (peek only after trying)

In [ ]:
class LSTMCellSolution(nn.Module):
    def __init__(self, in_dim, hid_dim, forget_bias=1.0):
        super().__init__()
        self.hid_dim = hid_dim
        self.W = nn.Linear(in_dim + hid_dim, 4 * hid_dim)
        with torch.no_grad():
            self.W.bias[hid_dim:2 * hid_dim].fill_(forget_bias)

    def forward(self, x, state):
        h, c = state
        z = self.W(torch.cat([h, x], dim=-1))
        i, f, g, o = z.chunk(4, dim=-1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        g = torch.tanh(g)
        o = torch.sigmoid(o)
        c_new = f * c + i * g
        h_new = o * torch.tanh(c_new)
        return h_new, c_new